# Part 0: Setup

In [2]:
# Install PEFT along with dependencies
!pip install -q peft transformers accelerate bitsandbytes

In [12]:
import os
from dotenv import load_dotenv
import torch
import platform
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import pandas as pd
from tqdm import tqdm


In [13]:
# RUN THIS CELL ONLY IF RUNNING ON PACE-ICE
# override the huggingface cache path and nltk cache path
dirs = {
    "HF_HOME":"~/scratch/hf_cache",
    "TRITON_CACHE_DIR":"~/scratch/triton_cache",
    "TORCHINDUCTOR_CACHE_DIR":"~/scratch/inductor_cache",
    'NLTK_DATA':"~/scratch/nltk_data"
}

for name in dirs:
    d = dirs[name]
    path = os.path.expanduser(d)
    print(name)
    print(path)
    os.makedirs(path, exist_ok=True)
    # making sure the cache dirs are rwx for owner
    os.chmod(path, 0o700)
    os.environ[name] = path
print("Make sure the cache files are in ~/scratch/ so quota doesn't exceed limit!")

HF_HOME
/home/hice1/yhsu72/scratch/hf_cache
TRITON_CACHE_DIR
/home/hice1/yhsu72/scratch/triton_cache
TORCHINDUCTOR_CACHE_DIR
/home/hice1/yhsu72/scratch/inductor_cache
NLTK_DATA
/home/hice1/yhsu72/scratch/nltk_data
Make sure the cache files are in ~/scratch/ so quota doesn't exceed limit!


In [ ]:
# Change this to your own token (or save in .env)
os.environ['HF_TOKEN'] = ''

In [15]:
load_dotenv()  # loads HF_TOKEN into environment
print("✅ Hugging Face token loaded from environment.")

✅ Hugging Face token loaded from environment.


In [16]:
print("=== 🧠 Environment Info ===")
print(f"Python version: {platform.python_version()}")
print(f"PyTorch version: {torch.__version__}")
print("-----------------------------")

# Check for CUDA (NVIDIA GPUs)
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"✅ CUDA is available. Number of GPUs: {num_gpus}")

    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        total_mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {gpu_name} ({total_mem:.2f} GB VRAM)")

    # Also show current GPU and free memory
    current_gpu = torch.cuda.current_device()
    print(f"\nUsing GPU: {torch.cuda.get_device_name(current_gpu)}")
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f"Available VRAM: {free_mem/1e9:.2f} GB / {total_mem/1e9:.2f} GB")

# Check for Apple Silicon (MPS)
elif torch.backends.mps.is_available():
    print("✅ Running on Apple Silicon (MPS backend).")

# Check for ROCm (AMD GPUs)
elif torch.version.hip is not None:
    print("✅ ROCm (AMD GPU) detected.")

# Otherwise fallback to CPU
else:
    print("⚠️ No GPU detected — running on CPU only.")
    print("This will be very slow for large models like Llama-3.1-8B.")

print("-----------------------------")

# Confirm torch default device
default_device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Default torch device: {default_device}")

=== 🧠 Environment Info ===
Python version: 3.10.13
PyTorch version: 2.8.0+cu128
-----------------------------
✅ CUDA is available. Number of GPUs: 2
  • GPU 0: NVIDIA H200 (139.80 GB VRAM)
  • GPU 1: NVIDIA H200 (139.80 GB VRAM)

Using GPU: NVIDIA H200
Available VRAM: 142.17 GB / 150.11 GB
-----------------------------
Default torch device: cuda


In [17]:

# --- 3. Model name on Hugging Face Hub ---
model_name = "meta-llama/Llama-3.1-8B"

# --- 4. (Optional) Authenticate if model is gated/private ---
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

print("Loading tokenizer and model…")

# --- 5. Load tokenizer ---
# Tokenizer converts text ↔ tokens. Must match model for correct vocabulary.
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # ensure padding works

# --- 6. Load model in full bf16 precision (no quantization) ---
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",      # Automatically distributes layers across GPUs
    torch_dtype=torch.bfloat16,  # Use bf16 for all layers
    low_cpu_mem_usage=True,       # Stream weights directly to GPU to reduce CPU RAM footprint
    trust_remote_code=True        # Needed if the repo includes custom code
)

print("✅ Model loaded successfully!")


Loading tokenizer and model…


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded successfully!


In [18]:

# --- 8. Simple inference test ---
prompt = """### Instruction:
Explain the difference between left-wing and right-wing economic policies.

### Response:"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=150,                  # control output length
    do_sample=True,                      # enables some randomness
    temperature=0.7,                     # mild creativity
    top_p=0.9,                           # nucleus sampling
    pad_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.2               # prevent repeated text
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

### Instruction:
Explain the difference between left-wing and right-wing economic policies.

### Response: 
* The left wing is about social justice. They support government intervention to help people out of poverty, ensure everyone has access to education and healthcare, etc.
* Right-wingers want less govt interference in economy and believe that if you work hard enough, you can succeed without govt assistance. 




In [23]:
# Base Test
cwd = os.getcwd()
data_path_eval = os.path.join(cwd, "sum_gen_val.jsonl")
eval_df = pd.read_json(data_path_eval, lines=True)
device = "cuda" if torch.cuda.is_available() else "cpu"


def build_eval_prompt(row):
    """
    Same format as training, but without including the output.
    The model should generate the part after '### Response:'.
    """
    return (
        f"### Instruction:\n{row['instruction'].strip()}\n\n"
        f"### Input:\n{row['input'].strip()}\n\n"
        f"### Response:\n"
    )


eval_prompts = [build_eval_prompt(r) for _, r in eval_df.iterrows()]
eval_references = [r["output"] for _, r in eval_df.iterrows()]  # gold answers
print("✅ Loaded val.jsonl successfully!")
print(f"Number of val samples: {len(eval_df)}\n")
print("📊 Preview:")
display(eval_df.head())


generated_outputs = []

for i, prompt in tqdm(enumerate(eval_prompts), total=len(eval_prompts), desc="Generating"):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    gen_tokens = output_ids[0, input_len:]
    gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    generated_outputs.append(gen_text)



✅ Loaded val.jsonl successfully!
Number of val samples: 713

📊 Preview:


,instruction,input,output
0,"Classify the following article as Left, Right,...",Headline: Education Federal Judge Says Bidens ...,"right, Summary: A federal judge ruled Thursday..."
1,"Classify the following article as Left, Right,...",Headline: 2024 Presidential Election How Are C...,"right, Summary: Last week’sreport from Special..."
2,"Classify the following article as Left, Right,...",Headline: Technology Twitter Users Vote Elon M...,"left, Summary: On Sunday, Elon Musk posted apo..."
3,"Classify the following article as Left, Right,...",Headline: Public Health Vinay Prasad Returns F...,"center, Summary: Dr. Vinay Prasad, Chief Offic..."
4,"Classify the following article as Left, Right,...",Headline: Who Did Voters Choose Tuesdays Prima...,"right, Summary: On Tuesday night,primary voter..."


Generating:   0%|          | 3/713 [00:17<1:07:28,  5.70s/it]


KeyboardInterrupt: 

In [ ]:

import json

with open("base_sum_gen_generated_outputs.jsonl", "w") as f:
    for prompt, output, expected_output in zip(eval_prompts, generated_outputs, eval_references):
        f.write(json.dumps({"prompt": prompt, "generation": output, "expected": expected_output}) + "\n")

print("Saved to base_sum_gen_generated_outputs.jsonl")

# Part 1: Load & Prepare Data
Data when loaded in:
```
{
    "instruction": "Write a political news story from a {left/ right/ center} perspective based on the headline.", 
    "input": "Headline: ...",
    "output": "..."
}
```

**Steps to process it:**
1. Combine into 1 traning text
2. Tokenize text

In [62]:
cwd = os.getcwd()

# build full paths
data_path = os.path.join(cwd, "sum_gen_train.jsonl")   # input file
data_path_eval = os.path.join(cwd, "sum_gen_val.jsonl")
# Option 1: Load using pandas — easiest for inspection
train_df = pd.read_json(data_path, lines=True)
eval_df = pd.read_json(data_path_eval, lines=True)
eval_prompts = [build_eval_prompt(r) for _, r in eval_df.iterrows()]
eval_references = [r["output"] for _, r in eval_df.iterrows()]  # gold answers
print("✅ Loaded train.jsonl & val.jsonl successfully!")
print(f"Number of train samples: {len(train_df)}\n Number of val samples: {len(eval_df)}\n")
print("📊 Preview:")
display(train_df.head())

✅ Loaded train.jsonl & val.jsonl successfully!
Number of train samples: 2852
 Number of val samples: 713

📊 Preview:


,instruction,input,output
0,"Classify the following article as Left, Right,...",Headline: If Democrats Flip House What Will Th...,"right, Summary: With Democrats largely project..."
1,"Classify the following article as Left, Right,...",Headline: Putin Marks Russias Victory Day Spee...,"center, Summary: On Monday, Russian President ..."
2,"Classify the following article as Left, Right,...",Headline: Media Industry Ap Criticized After D...,"left, Summary: The Associated Press (Lean Left..."
3,"Classify the following article as Left, Right,...",Headline: Economy And Jobs Will There Be Reces...,"left, Summary: Since mid-2022, economists and ..."
4,"Classify the following article as Left, Right,...",Headline: Technology Plans Tesla Tunnel Nashvi...,"right, Summary: Tennessee Gov. Bill Lee announ..."


In [63]:
def format_prompt(example):
    """
    Combine instruction, input, and output into one training text.
    """
    return (
        f"### Instruction:\n{example['instruction'].strip()}\n\n"
        f"### Input:\n{example['input'].strip()}\n\n"
        f"### Response:\n{example['output'].strip()}"
    )
def build_eval_prompt(row):
    """
    Same format as training, but without including the output.
    The model should generate the part after '### Response:'.
    """
    return (
        f"### Instruction:\n{row['instruction'].strip()}\n\n"
        f"### Input:\n{row['input'].strip()}\n\n"
        f"### Response:\n"
    )

In [67]:
eval_prompts = [build_eval_prompt(r) for _, r in eval_df.iterrows()]
eval_references = [r["output"] for _, r in eval_df.iterrows()]  # gold answers
print(eval_prompts[0][:400])
print("-------")
print(eval_references[0][:400])


### Instruction:
Classify the following article as Left, Right, or Neutral. Write an unbiased summary for the article.

### Input:
Headline: Education Federal Judge Says Bidens Student Loan Forgiveness Plan Illegal, Story: National Review

Senate Passes Bill to End Longest Government Shutdown in History

Trump HHS Nixes Biden-Era Rule Incentivizing Doctors to Formulate ‘Anti-Racism’ Plans

Knives 
-------
right, Summary: A federal judge ruled Thursday that President Joe Biden's student loan forgiveness plan is illegal.Key Quotes:In his decision, U.S. District Judge Mark T. Pittman of the Northern District of Texaswrotethat "the HEROES Act—a law to provide loan assistance to military personnel defending our nation—does not provide the executive branch clear congressional authorization to create a $4


In [65]:
# Apply formatting to DataFrames
train_df["text"] = train_df.apply(format_prompt, axis=1)

# Preview one formatted sample
print(train_df["text"].iloc[0][:800])  # show first 800 chars

### Instruction:
Classify the following article as Left, Right, or Neutral. Write an unbiased summary for the article.

### Input:
Headline: If Democrats Flip House What Will They Prioritize, Story: Quotes displayed in real-time or delayed by at least 15 minutes. Market data provided byFactset.
          Powered and implemented byFactSet Digital Solutions.Legal Statement.

This material may not be published, broadcast, rewritten, or redistributed. ©2025 FOX News Network, LLC. All rights reserved.FAQ-New Privacy Policy

If the Democrats win a majority in theHouse, it will be Democrats who chair key committees. And since the chair is picked largely on the basis of seniority, we know who they will be.

So let me introduce the Democrats who will become very powerful, if the Democrats win next 


In [66]:
from datasets import Dataset
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=1024,   # adjust based on GPU memory
        padding="max_length"
    )
# Convert to HF Dataset and tokenize
train_dataset = Dataset.from_pandas(train_df[["text"]])
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Check shape and sample
print(tokenized_train)

print("============ Sanity Check - decoding encoded tokens ============")
print(tokenizer.decode(tokenized_train[0]["input_ids"][:200]))

Map:   0%|          | 0/2852 [00:00<?, ? examples/s]

Dataset({
    features: ['__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 2852
})
============ Sanity Check - decoding encoded tokens ============
<|begin_of_text|>### Instruction:
Classify the following article as Left, Right, or Neutral. Write an unbiased summary for the article.

### Input:
Headline: If Democrats Flip House What Will They Prioritize, Story: Quotes displayed in real-time or delayed by at least 15 minutes. Market data provided byFactset.
          Powered and implemented byFactSet Digital Solutions.Legal Statement.

This material may not be published, broadcast, rewritten, or redistributed. ©2025 FOX News Network, LLC. All rights reserved.FAQ-New Privacy Policy

If the Democrats win a majority in theHouse, it will be Democrats who chair key committees. And since the chair is picked largely on the basis of seniority, we know who they will be.

So let me introduce the Democrats who will become very powerful, if the Democrats win next week.

Maxine Wate

# Part 2: LoRA fine-tuning
**Steps**
1. Set up LoRA model
2. Set up training arguments
3. Prepare a Data Collator
4. Set up Trainer
5. Run Training
6. Save new weights

In [49]:
# Set up Lora model for fine-tuning
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,                # rank of the LoRA matrices
    lora_alpha=32,       # scaling factor
    target_modules=["q_proj", "v_proj"],  # which layers to fine-tune
    lora_dropout=0.05,   # dropout for LoRA
    bias="none",         # keep bias frozen
    task_type="CAUSAL_LM" # type of task
)

# Wrap base model with PEFT
model = get_peft_model(model, lora_config) # freezes original layer
model.print_trainable_parameters()  # confirm only LoRA params are trainable

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


/home/hice1/yhsu72/.local/lib/python3.10/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/hice1/yhsu72/.local/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [50]:
# Define Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="outputs/sum_gen_lora",         # where to save checkpoints & LoRA adapters
    per_device_train_batch_size=8,     # batch size per GPU
    gradient_accumulation_steps=4,     # effective batch size = 8 * 4 * 2 GPUs = 64
    learning_rate=3e-4,                # LoRA-friendly default
    num_train_epochs=8,                # 3 epochs, increase if dataset is larger
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,                          # optional: H200 supports bf16, can improve speed
    dataloader_drop_last=True,           # drop incomplete batch to avoid OOM
    report_to="none",                    # change to 'wandb' if using W&B
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,   # improves multi-GPU performance
    gradient_checkpointing=True,        # reduce memory usage for large models
    warmup_steps=50,                    # optional warmup
    optim="paged_adamw_32bit",          # memory-efficient optimizer
    lr_scheduler_type="cosine"          # smooth LR schedule
)


print("✅ Training arguments set up")


✅ Training arguments set up


In [51]:
# Prepare Data Collator (Coverts list of examples into tensors)
from transformers import DataCollatorForLanguageModeling

# Collator for causal LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # False because this is causal LM, not masked LM
)

print("✅ Data collator ready")

✅ Data collator ready


In [52]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("✅ Trainer ready")

✅ Trainer ready


In [53]:
# Begin training
trainer.train()

# Save the LoRA adapter after training
model.save_pretrained("outputs/sum_gen_lora_adapter")

print("✅ Training complete and LoRA adapter saved")


Step,Training Loss
50,1.923000
100,1.786200
150,1.728000
200,1.698400
250,1.681000
300,1.636500
350,1.636300
400,1.607900
450,1.586100
500,1.558200


✅ Training complete and LoRA adapter saved


# Part 3: Quick Tests

In [54]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, os

# ---- Paths ----
# base model you used during LoRA training
base_model_name = "meta-llama/Llama-3.1-8B"   # change if you used a different one

# absolute path to your LoRA adapter (from the screenshot)
adapter_path = "./outputs/sum_gen_lora_adapter"

print("Base model name:", base_model_name)
print("Adapter path:", adapter_path, "exists?", os.path.isdir(adapter_path))

# ---- Load tokenizer from base model ----
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

# ---- Load base model ----
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# ---- Attach LoRA adapter on top of base model ----
trained_model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
)

# Optional: merge LoRA weights into the base model and drop the adapter structure
# trained_model = trained_model.merge_and_unload()

print("✅ Loaded base model + LoRA adapter; ready for evaluation!")


Base model name: meta-llama/Llama-3.1-8B
Adapter path: ./outputs/sum_gen_lora_adapter exists? True


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Loaded base model + LoRA adapter; ready for evaluation!


/home/hice1/yhsu72/.local/lib/python3.10/site-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.2.sel

In [55]:
from datetime import datetime
# --- Test generation ---
instruction = "Classify the following article as Left, Right, or Neutral. Write an unbiased summary for the article."

test_input = "Headline: 2024 Presidential Election How Are Concerns Over Bidens Age Impacting His Reelection Chances, Story: National Review\n\nWhat’s the Right’s Answer to Mamdani’s ‘Affordability’ Challenge?\n\nThe Shutdown Was Pointless and Dumb\n\nThe Government’s Shameful Stake in the Gambling Glut\n\nEnd the Paperwork Tax on Investors\n\nDemocrats Caved on the Shutdown\n\nThereis an element of Scooby-Dooism to the Democrats’ erroneous conviction that, if they just insist vehemently enough that Joe Biden is not, in fact, clearly too old to be president of these United States, they will be able to persuade the public that it is true. “We would have got away with it,” Biden’s apologists seem to be muttering aloud, “if it hadn’t been for that pesky Robert Hur!”\n\nWhich, of course, is quite inordinately silly. There is no chance that the Democrats will be able to get around Joe Biden’s obvious decline, given that the people to whom they...\n\nContinue reading this article with an NRPLUS subscription.\n\nEnjoy full access for 60% off.\n\nAlready a member?Sign in.\n\nDel Toro’s remake refuses to grow up.\n\nThe longer Congress writes laws that read like mission statements, the more frustrated voters will be when the mission goes sideways.\n\nThe Sierra Club is a cautionary tale.\n\nFor a few years, the all-volunteer military construct was failing. Fresh thinking, financial resources, and untold amounts of effort have turned things around.\n\nThe market works, but it needs champions.\n\nThe real reason for the shutdown? It was a way for progressives to give vent to an unreasoning hatred of Donald Trump.\n\n© 2025 National Review\n\nNewsletters\n\n© 2025 National Review"

test_output = "right, Summary: Last week’sreport from Special Counsel Robert Hurrenewed debate regarding President Joe Biden’s age and ability to win reelection and lead the country for an additional four years.From the Left:David Rothkopf (Lean Left bias)wrote that Democrats need a “reality check.” Instead of denying Biden’s advanced age, Rothkopf calls for Biden’s team to “own it,” writing, “Let’s see more of him not in staged settings, but in the kind of informal, unstaged videos that matter most today on Instagram, YouTube, or TikTok.” A writer inVox (Left bias)outlined the process of replacing Biden as the Democratic nominee, determining it to be a “herculean, if not impossible, task” and concluding, “If you’re a Democrat, or want to beat Donald Trump, it looks like it’s Biden or bust.”From the Right:Charles C. W. Cooke (Right bias)argued that there is “no chance that the Democrats will be able to get around Joe Biden’s obvious decline.” Outlining why the Democrat’s current strategy won’t work moving forward, Cooke concluded, “President Biden cannot fake being competent because the people who are ultimately empowered to judge his competence have no great obligation to pretend that they cannot see the truth.” A writer in theWashington Examiner (Lean Right bias)echoed concerns, writing, “None of us gets out of here alive, and 81-year-old Biden is not going to level out onto a plateau of acceptable lucidity. He is going to step down again and again until he reaches the cellar door."
prompt = f"""### Instruction:
{instruction}
### Input:
{test_input}
### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output_ids = trained_model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated_text)


# ------------------------
# SAVE GENERATED TEXT
# ------------------------

# Folder to save the generated files
save_dir = "generated_outputs_sum_gen"
os.makedirs(save_dir, exist_ok=True)

# Create a filename using timestamp
filename = f"output_right_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
file_path = os.path.join(save_dir, filename)

# Save the text
with open(file_path, "w", encoding="utf-8") as f:
    f.write(generated_text)

print(f"\nSaved generated text to: {file_path}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


### Instruction:
Classify the following article as Left, Right, or Neutral. Write an unbiased summary for the article.
### Input:
Headline: 2024 Presidential Election How Are Concerns Over Bidens Age Impacting His Reelection Chances, Story: National Review

What’s the Right’s Answer to Mamdani’s ‘Affordability’ Challenge?

The Shutdown Was Pointless and Dumb

The Government’s Shameful Stake in the Gambling Glut

End the Paperwork Tax on Investors

Democrats Caved on the Shutdown

Thereis an element of Scooby-Dooism to the Democrats’ erroneous conviction that, if they just insist vehemently enough that Joe Biden is not, in fact, clearly too old to be president of these United States, they will be able to persuade the public that it is true. “We would have got away with it,” Biden’s apologists seem to be muttering aloud, “if it hadn’t been for that pesky Robert Hur!”

Which, of course, is quite inordinately silly. There is no chance that the Democrats will be able to get around Joe Biden

# Section 4 Evaluation

In [68]:
print(len(eval_prompts))

713


In [69]:
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
trained_model.to(device)

generated_outputs = []

for i, prompt in tqdm(enumerate(eval_prompts), total=len(eval_prompts), desc="Generating"):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = trained_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    gen_tokens = output_ids[0, input_len:]
    gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    generated_outputs.append(gen_text)


Generating: 100%|██████████| 713/713 [28:21<00:00,  2.39s/it]


In [76]:
import json

with open("sum_gen_generated_outputs.jsonl", "w") as f:
    for prompt, output, expected_output in zip(eval_prompts, generated_outputs, eval_references):
        f.write(json.dumps({"prompt": prompt, "generation": output, "expected": expected_output}) + "\n")

print("Saved to sum_gen_generated_outputs.jsonl")


Saved to sum_gen_generated_outputs.jsonl


In [35]:
# Load in the evaluation results
eval_prompts = []
generated_outputs = []
eval_references = []
file = "base_sum_gen_generated_outputs.jsonl"
with open(file) as f:
    for line in f:
        item = json.loads(line)
        
        p = item["prompt"]
        o = item["generation"]
        e = item["expected"]
        
        eval_prompts.append(p)
        generated_outputs.append(o)
        eval_references.append(e)
print(f"loaded {file}")

loaded base_sum_gen_generated_outputs.jsonl


Distinct N evaluation

In [36]:
from collections import Counter

def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def distinct_n(corpus, n=1):
    all_ngrams = []
    for text in corpus:
        tokens = text.split()
        if len(tokens) < n:
            continue
        all_ngrams.extend(get_ngrams(tokens, n))

    if not all_ngrams:
        return 0.0

    unique_ngrams = set(all_ngrams)
    return len(unique_ngrams) / len(all_ngrams)

d1 = distinct_n(generated_outputs, n=1)
d2 = distinct_n(generated_outputs, n=2)

print(f"Distinct-1: {d1:.4f}")
print(f"Distinct-2: {d2:.4f}")


Distinct-1: 0.1327
Distinct-2: 0.3989


In [29]:
prompt_texts = [str(x) for x in eval_prompts]

In [37]:
from sentence_transformers import SentenceTransformer, util

sem_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

emb_prompts = sem_model.encode(prompt_texts, convert_to_tensor=True)
emb_gen     = sem_model.encode(generated_outputs, convert_to_tensor=True)

cos_scores = util.cos_sim(emb_prompts, emb_gen).diagonal()

avg_sim = float(cos_scores.mean())
print("First 10 prompt↔generation similarities:",
      [round(float(x), 4) for x in cos_scores[:10]])
print(f"\nAverage semantic similarity: {avg_sim:.4f}")

First 10 prompt↔generation similarities: [0.6128, 0.0626, 0.7594, 0.1736, 0.5748, 0.0301, 0.5371, 0.8624, 0.1425, 0.68]

Average semantic similarity: 0.4164


In [80]:
output_path = "sum_gen_eval_results.jsonl"
print("Saving to:", os.path.abspath(output_path))

with open(output_path, "w", encoding="utf-8") as f:
    for prompt, generation, sim in zip(eval_prompts, generated_outputs, cos_scores):
        record = {
            "prompt": prompt,
            "generation": generation,
            "semantic_similarity": float(sim),
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved", len(generated_outputs), "records with similarity to", output_path)


Saving to: /storage/ice1/5/3/yhsu72/CS_4650_Project/training/sum_gen_eval_results.jsonl
Saved 713 records with similarity to sum_gen_eval_results.jsonl


In [31]:
import json

# Categories to check
CATEGORIES = ["left", "right", "center", "neutral"]

total = 0
valid = 0
correct = 0

with open("base_sum_gen_generated_outputs.jsonl") as f:
    for line in f:
        total += 1
        item = json.loads(line)

        gen = item["generation"].lower()
        expected = item["expected"].lower()[:20]
        

        # 1. First 20 characters
        first20 = gen[:20]

        # 2. Count categories that appear in first 20 chars
        present = [cat for cat in CATEGORIES if cat in first20]

        if len(present) != 1:
            # invalid output: either none or more than one
            continue

        valid += 1
        predicted = present[0]

        # 3. Compare with expected tag

        if predicted in expected:
            correct += 1


# Final statistics
percent_valid = valid / total if total > 0 else 0
percent_correct = correct / valid if valid > 0 else 0

print(f"Total outputs: {total}")
print(f"Valid outputs: {valid}")
print(f"Correct predictions: {correct}")
print(f"% Valid generation: {percent_valid:.4f}")
print(f"% Right guess: {percent_correct:.4f}")

Total outputs: 713
Valid outputs: 412
Correct predictions: 182
% Valid generation: 0.5778
% Right guess: 0.4417
